# L6 — CloudZero AI Signals: Model-Call Usage

## What you'll learn

- How the Order, Refund and Research agents send one span per model call to CloudZero
- How to check that your CloudZero API key and username are saved and valid, without showing the key
- How to find **your own** model calls in the CloudZero **AI Signals → Livestream** view, and confirm them by their **Attendee Id**
- What CloudZero mode covers, and what it does not (the orchestrator Harness)

## Overview

When you selected **CloudZero** as the observability provider in the Partner Selection UI, [`2_specialist_agents.ipynb`](../l3-orchestration/2_specialist_agents.ipynb) gave the three specialist agents a small extra exporter (`agents/cz_otel.py`). Each time a specialist calls its model through LiteLLM, LiteLLM emits one `litellm_request` span with the model, token counts and cost, and **no prompt or completion text**. The exporter sends only those spans straight to CloudZero's telemetry intake. CloudZero shows them as **AI Signals** events.

Everything else is unchanged from CloudWatch mode: the managed ADOT stays on, so **all traces and all logs still go to Amazon CloudWatch**, and CloudWatch GenAI Observability and the L7 notebooks keep working.

Each span carries a resource attribute `attendee_id` set to the CloudZero username you saved in L0. Every attendee sends to the same workshop org, so you find your own events by the time you ran Step 3 and the model, then confirm the **Attendee Id** in the event's details.

## Architecture

![Observability layer: the agents emit OpenTelemetry spans. By default they go through managed ADOT to CloudWatch and CloudWatch Omni; with the Datadog option an OTLP exporter in the agent sends them to Datadog Agent Observability.](images/l6-observability-architecture.png)

CloudZero mode is not drawn in this diagram. It follows the CloudWatch path at the top: all traces and logs go to CloudWatch. In addition, the Order, Refund and Research agents send their model-call spans to CloudZero, as shown below.

```
                         Chatbot (L5) / Step 3 below
                                     |
                    Orchestrator Harness (anycompany-orchestrator)
                         |   NOT sent to CloudZero (traces -> CloudWatch)
       delegates via     |
       AgentCore Gateway v
   +-----------------------------------------------+
   | Order    Runtime (anycompany-order-agent)     |   LiteLLM litellm_request spans only
   | Refund   Runtime (anycompany-refund-agent)    | ---------------------------------------> CloudZero
   | Research Runtime (anycompany-research-agent)  |   private TracerProvider (agents/cz_otel.py)
   +-----------------------------------------------+   resource: attendee_id=<your username>
            |                                             -> AI Signals
            +--> all traces (managed ADOT) and all logs --> Amazon CloudWatch
```

## Steps

0. Install dependencies
1. Load config and check your CloudZero credentials (key never shown)
2. How CloudZero fits next to CloudWatch
3. Generate traffic through the deployed agents
4. Optional: confirm the agents started the CloudZero exporter
5. Sign in to CloudZero
6. Find your model calls in AI Signals → Livestream
7. Open an event: Attendee Id, model, tokens
8. Limitations

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- Your CloudZero **username**, **password** and **API key** from the workshop vending page: `https://<CZ_VENDING_PAGE>` (the **Get your CloudZero API key & username** link on the CloudZero card in the Partner Selection UI).
- In the Partner Selection UI (**Orchestration → Agent Observability**) you clicked the **CloudZero** card, entered the API key and the username exactly as the vending page shows them in the **CloudZero API key** and **CloudZero username** fields, and clicked **Save to SSM**. That writes:
  - `/anycompany/agentcore/config/observability/provider` = `cloudzero` (String)
  - `/anycompany/agentcore/config/cloudzero/api_key` (**SecureString**)
  - `/anycompany/agentcore/config/cloudzero/username` (String)
- Step 6 of [`1_gateway_tools.ipynb`](../l3-orchestration/1_gateway_tools.ipynb) confirmed CloudZero, and **after that** you deployed the L3 agents ([`2_specialist_agents.ipynb`](../l3-orchestration/2_specialist_agents.ipynb) and [`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb)). Agents keep the backend they were deployed with: if you picked CloudZero after deploying, re-run both notebooks.
- `harness_arn` is in SSM (Step 5 of `3_orchestrator_agent.ipynb` saves it).

## Step 0: Install Dependencies

`boto3` reads SSM and CloudWatch Logs and invokes the Harness. Nothing CloudZero-specific is installed: this notebook never calls CloudZero itself. The install is idempotent - safe to re-run.

In [ ]:
%pip install --quiet boto3

## Step 1: Load config and check your CloudZero credentials

This cell confirms that CloudZero is the selected provider, that both CloudZero parameters exist, and that they pass **the same validation the deploy notebooks run** (it imports `cloudzero_otel_env` from `l3-orchestration/helpers/observability.py`). The validation rejects:

- an empty API key, or one with whitespace, a control character, a non-ASCII character or `,`
- an empty username, or one with a control character, `,` or `%`

The API key is **never printed**: you see only that it is present, its parameter type and its length. The username is printed because it is the **Attendee Id** you will look for in CloudZero.

In [ ]:
import os
import sys
from pathlib import Path

import boto3

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
SSM_PREFIX = "/anycompany/agentcore"
CZ_PREFIX = f"{SSM_PREFIX}/config/cloudzero"

# Reuse the L3 helper so this check is exactly the one the deploy notebooks run.
# Run this notebook from aws-reinvent/workshop/l6-observability/.
L3_DIR = (Path.cwd().parent / "l3-orchestration").resolve()
if not (L3_DIR / "helpers" / "observability.py").is_file():
    raise RuntimeError(
        f"{L3_DIR}/helpers/observability.py not found. Open this notebook from "
        "aws-reinvent/workshop/l6-observability/ so the L3 helpers can be imported."
    )
if str(L3_DIR) not in sys.path:
    sys.path.insert(0, str(L3_DIR))
from helpers.observability import get_observability_provider, cloudzero_otel_env

ssm = boto3.client("ssm", region_name=REGION)


def _get(name, decrypt=False):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=decrypt)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound as _e:
        raise RuntimeError(
            f"{name} not found in SSM. Run the l3-orchestration notebooks (1_gateway_tools, "
            "2_specialist_agents and 3_orchestrator_agent) with CloudZero selected first."
        ) from _e


PROVIDER = get_observability_provider(ssm, SSM_PREFIX)   # "cloudwatch" if none selected
if PROVIDER != "cloudzero":
    raise RuntimeError(
        f"Observability provider is '{PROVIDER}', not 'cloudzero'. This is the CloudZero "
        "notebook - if you chose CloudWatch in L0, run 1_cloudwatch_omni.ipynb; for Datadog, "
        "run 1_datadog_agent_observability.ipynb."
    )

# 1. Both parameters exist. Read without decryption: only the name and type are shown.
missing = []
for name in (f"{CZ_PREFIX}/api_key", f"{CZ_PREFIX}/username"):
    try:
        ptype = ssm.get_parameter(Name=name)["Parameter"]["Type"]
        print(f"  found   {name}  ({ptype})")
    except ssm.exceptions.ParameterNotFound:
        missing.append(name)
        print(f"  MISSING {name}")
if missing:
    raise RuntimeError(
        "CloudZero is selected but its API key or username is missing. Save them in the "
        "Partner Selection UI (Orchestration -> Agent Observability), then re-run this cell."
    )

# 2. Same validation as the deploy notebooks. A ValueError here names the parameter
#    and the problem, never the key.
_cz = cloudzero_otel_env(ssm, "anycompany-check", SSM_PREFIX)
CZ_USERNAME = _cz["CZ_ATTENDEE_ID"]
CZ_ENDPOINT = _cz["CZ_OTEL_ENDPOINT"]
_key_len = len(_cz["CZ_API_KEY"])
del _cz  # drop the decrypted key from the notebook namespace

print()
print("CloudZero configuration is valid.")
print(f"  provider:     {PROVIDER}")
print(f"  api_key:      present, {_key_len} characters (value not shown)")
print(f"  username:     {CZ_USERNAME}")
print(f"  attendee_id:  {CZ_USERNAME}   <- shown as Attendee Id in CloudZero event details")
print(f"  ingest:       {CZ_ENDPOINT}")
if _key_len != 38:
    print()
    print(f"  Note: workshop API keys are 38 characters; this one is {_key_len}. If you pasted it")
    print("  by hand, check it against the vending page and save it again in Partner Selection.")

## Step 2: How CloudZero fits next to CloudWatch

CloudZero mode **adds** a usage view of model calls. It does not replace CloudWatch.

| | CloudWatch (always on) | CloudZero AI Signals |
|---|---|---|
| **Where you work** | AWS Console / CloudWatch GenAI Observability / CloudWatch Omni | CloudZero app at `https://app.cloudzero.com`, org **re:Invent Agentic AI Lab** |
| **What arrives** | All traces from all four services (managed ADOT) | Only LiteLLM `litellm_request` spans, one per model call, from the Order, Refund and Research agents |
| **What a record holds** | Full span tree, tools, timing | Model, input/output tokens, estimated cost. No prompt or completion text |
| **Orchestrator Harness** | Yes | **No** (see Limitations) |
| **How you find your data** | Session id, service name | Time of your run and the model; confirm **Attendee Id** (= your CloudZero username) in Event details |
| **Logs** | CloudWatch | **Still CloudWatch** - CloudZero receives no logs |

How the spans get there (wired once in L3, nothing to set here):

- Ingest: `POST https://telemetry.cloudzero.com/v2/telemetry/litellm/v1/traces`, OTLP/HTTP protobuf, with your API key as the `Authorization` header.
- CloudZero recognises the spans by their name, `litellm_request`.
- Resource attributes become CloudZero dimensions, except `service.*`, `deployment.environment*` and `telemetry.sdk.*`. The exporter sends only `service.name` and `attendee_id`, so `attendee_id` is the per-attendee tag (shown as **Attendee Id** in an event's details). `service.name` (which agent made the call) is not a dimension.
- A CloudZero setup problem never stops an agent: it logs an error (without the key) and keeps serving, untraced in CloudZero. Step 4 shows you how to spot that.

## Step 3: Generate traffic through the deployed agents

Invoke the Harness several times in a single session with questions that make it delegate to each specialist at least once:

- Turn 1 - an **order lookup** -> routes to `anycompany-order-agent`
- Turn 2 - a **refund request** -> routes to `anycompany-refund-agent`
- Turn 3 - a **product research** question -> routes to `anycompany-research-agent` (only if you deployed the optional Research Agent in `2_specialist_agents.ipynb`)

Every model call a specialist makes produces one CloudZero event tagged with your `attendee_id`. The orchestrator's own model calls are **not** sent (see Limitations), so you will see only the specialists' calls. The cell prints the UTC time window of the run: you use it in Step 6 to pick out your events in CloudZero.

In [ ]:
import time
import uuid
from datetime import datetime, timezone
from botocore.config import Config

ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
HARNESS_ARN = _get(f"{SSM_PREFIX}/harness_arn")

# invoke_harness requires runtimeSessionId to be 33-100 chars. Pad a readable
# prefix with a uuid4 hex.
session_id = f"cz-demo-{int(time.time())}-{uuid.uuid4().hex}"
actor_id = f"workshop-user-{ACCOUNT_ID[-4:]}"
print(f"Session ID:  {session_id}")
print(f"Actor ID:    {actor_id}")
print(f"attendee_id: {CZ_USERNAME}")

runtime = boto3.client("bedrock-agentcore", region_name=REGION, config=Config(read_timeout=300))

# Three turns that make the Harness delegate to each specialist. Only the
# specialists' model calls reach CloudZero.
QUESTIONS = [
    "Look up order ORD-10001 for customer CUST-789.",
    "I want to refund ORD-10001 because it arrived damaged - am I eligible?",
    "Can you research the best alternative wireless headphones I could buy instead?",
]

RUN_START = datetime.now(timezone.utc)
for turn, question in enumerate(QUESTIONS, start=1):
    print()
    print("=" * 72)
    print(f"Turn {turn}: {question}")
    print("=" * 72)
    response = runtime.invoke_harness(
        harnessArn=HARNESS_ARN,
        runtimeSessionId=session_id,
        actorId=actor_id,
        messages=[{"role": "user", "content": [{"text": question}]}],
    )
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
    print()
RUN_END = datetime.now(timezone.utc)

print()
print("-" * 72)
print("Done. In CloudZero: AI Signals -> Livestream. Look for source litellm in this window,")
print(f"then open an event and check that Attendee Id = {CZ_USERNAME}")
print(f"Run window (UTC): {RUN_START:%Y-%m-%d %H:%M:%S} -> {RUN_END:%Y-%m-%d %H:%M:%S}")

## Step 4 (optional): Confirm the agents started the CloudZero exporter

This step **sends nothing new** to CloudZero. It reads the specialists' CloudWatch Logs for the one line `agents/cz_otel.py` writes when an agent starts:

- `CloudZero tracing enabled: LiteLLM model-call spans -> <endpoint> (service.name=..., attendee_id=...)` - the exporter is installed in that agent.
- `CloudZero tracing disabled: <reason>` - the agent is serving but sending nothing to CloudZero. Fix the value it names in Partner Selection, then re-run `2_specialist_agents.ipynb`.
- No line at all - usually the agent was deployed before CloudZero was selected (re-run `2_specialist_agents.ipynb`), or its logs have not arrived yet (wait a minute and re-run).

Neither line contains the API key. You may also see `Proxy Server is not installed. Skipping OpenTelemetry initialization.` in the same logs: it is harmless and refers to the LiteLLM proxy, which these agents do not use.

**What this does not prove.** The ingest endpoint answers `202 Accepted` even when it later drops data, so neither a `202` nor the "enabled" line proves your spans landed. The only proof is seeing the events in CloudZero (Step 6). Events can take a few minutes to appear in AI Signals; wait and refresh before assuming something is wrong.

In [ ]:
# OPTIONAL, read-only: look for the CloudZero startup line in each specialist's
# runtime logs. Nothing is sent to CloudZero.
from datetime import timedelta

logs = boto3.client("logs", region_name=REGION)
SINCE_HOURS = 24
start_ms = int((datetime.now(timezone.utc) - timedelta(hours=SINCE_HOURS)).timestamp() * 1000)

SPECIALISTS = [
    ("order", "order_agent_runtime_arn"),
    ("refund", "refund_agent_runtime_arn"),
    ("research", "research_agent_runtime_arn"),   # optional agent
]

for label, ssm_key in SPECIALISTS:
    try:
        arn = ssm.get_parameter(Name=f"{SSM_PREFIX}/{ssm_key}", WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        print(f"{label:<9} not deployed ({ssm_key} not in SSM) - skipped")
        continue
    group = f"/aws/bedrock-agentcore/runtimes/{arn.split('/')[-1]}-DEFAULT"
    events = []
    try:
        pages = logs.get_paginator("filter_log_events").paginate(
            logGroupName=group, startTime=start_ms, filterPattern='"CloudZero tracing"')
        for page in pages:
            events.extend(page.get("events", []))
    except logs.exceptions.ResourceNotFoundException:
        print(f"{label:<9} no log group yet ({group}) - invoke the agent, then re-run")
        continue
    if not events:
        print(f"{label:<9} no 'CloudZero tracing' line in the last {SINCE_HOURS} h")
        continue
    last = max(events, key=lambda e: e["timestamp"])
    when = datetime.fromtimestamp(last["timestamp"] / 1000, tz=timezone.utc)
    msg = last["message"].strip()
    status = "ENABLED " if "tracing enabled" in msg else "DISABLED"
    print(f"{label:<9} {status} at {when:%Y-%m-%d %H:%M:%S} UTC")
    print(f"          {msg[:300]}")

## Step 5: Sign in to CloudZero

1. Open `https://app.cloudzero.com`. It redirects you to the sign-in page at `auth.cloudzero.com`.
2. Sign in with the **username** and **password** from the vending page (`https://<CZ_VENDING_PAGE>`).
3. You land on **Explorer**. The organization switcher at the top of the left sidebar shows **re:Invent Agentic AI Lab**, the shared workshop org every attendee sends to.

Explorer is CloudZero's cloud cost view. Billing data is not loaded into the workshop org, so Explorer may show no data or keep loading; that is expected and does not affect this lab. Your model calls are under **AI Signals** (next step).

![screenshot: CloudZero Explorer after sign-in, org switcher showing re:Invent Agentic AI Lab](images/cloudzero_signin_home.png)

## Step 6: Find your model calls in AI Signals → Livestream

1. In the left sidebar, under **TOOLS & APPS**, click **AI Signals**.
2. AI Signals has three tabs: **Overview | Livestream | AI Explorer**. Click **Livestream** (direct link: `https://app.cloudzero.com/ai-signals/livestream`).
3. Livestream shows summary tiles (**Token usage**, **Cache**, **Cost**, **Models**) and a table of events with the columns **Timestamp**, **Caller**, **Model**, **Source**, **Tokens**, **Cache** and **Cost**. Above the table are the **All Models** and **All Sources** filters, a **Search events...** box and **Customize Table**.

Each model call from your specialist agents is one row with **Source** `litellm`. **Caller** shows `Unknown`; that is expected, because the exporter sends no caller information.

**Finding your own rows.** The workshop org is shared, and Livestream cannot filter or search by attendee: the search box does not match the Attendee Id. Pick out your events instead by:

- **Timestamp** - within the UTC run window printed at the end of Step 3 (timestamps are shown in UTC)
- **Model** - the model your specialists use (`/anycompany/agentcore/model_id` in SSM); you can narrow the table with **All Models**
- **Source** - `litellm` (you can narrow with **All Sources**)

Then open a row (Step 7) and confirm that its **Attendee Id** is the username printed in Step 1.

What to expect:

- One row per specialist model call. A single question usually takes more than one model call (the agent calls a tool, then answers), so expect several rows per turn.
- Only the Order, Refund and Research agents appear. The orchestrator's model calls are not sent.
- Rows can take a few minutes to appear. If yours are not there yet, wait and refresh.

![screenshot: CloudZero AI Signals Livestream tab with a litellm event](images/cloudzero_ai_signals_list.png)

## Step 7: Open an event: Attendee Id, model, tokens

Click a row in Livestream. The **Event details** panel opens and shows, from top to bottom:

| Section | Fields |
|---|---|
| Summary | **Caller**, **Model**, **Source**, **Events** |
| **Activity** | **Attendee Id** - your CloudZero username (the `attendee_id` your agents send) |
| **Token usage** | **Input tokens**, **Output tokens** |
| **Cost & performance** | **Est. cost**, **Avg latency**, **Error count** |
| | Timestamps and a **Raw payload** |

If **Attendee Id** matches the username from Step 1, the event is yours. The event carries **no prompt or completion text**: message logging is turned off in the exporter.

**About cost:** the cost for an event may show as `$0.00` (Est. cost) or `N/A` (the Cost column). Use the token counts as the reliable measure of usage in this lab.

![screenshot: CloudZero AI Signals Event details panel with Attendee Id, tokens and cost](images/cloudzero_ai_signals_event.png)

## Step 8: Limitations

- **The orchestrator Harness is not covered.** It runs LiteLLM inside the managed AgentCore Harness, which gives us no entry point to register LiteLLM's OpenTelemetry callback. Its model calls, which are often the largest share of a conversation's tokens, do not appear in CloudZero. Its traces stay on CloudWatch.
- **Only LiteLLM model-call spans are sent.** CloudZero receives `litellm_request` spans and nothing else: no Strands agent or tool spans, no A2A spans, no runtime spans, no logs. To see a whole conversation, use CloudWatch (`1_cloudwatch_omni.ipynb` or CloudWatch GenAI Observability).
- **No per-agent split in CloudZero.** The only per-attendee attribute the exporter adds is `attendee_id`, and Livestream cannot filter by it: find your rows by time and model, then confirm the Attendee Id.
- **Cost may show as `$0.00` or `N/A`.** Rely on the token counts.
- **A `202` from ingest is not proof of landing.** Check AI Signals itself.
- **The agents keep the backend they were deployed with.** If you change provider or credentials in Partner Selection, re-run `2_specialist_agents.ipynb` (and `3_orchestrator_agent.ipynb`) so the specialists pick it up.

## Summary

| What you checked | Where |
|---|---|
| CloudZero is the provider; API key and username saved and valid (key not shown) | Step 1, SSM `config/cloudzero/*` |
| Specialists' model calls generated | Step 3, `invoke_harness` |
| Exporter started in each specialist | Step 4, CloudWatch Logs (`CloudZero tracing ...`) |
| Your model calls, with model and tokens | CloudZero **AI Signals → Livestream**; Event details **Attendee Id** = your username |

What you did:
- Confirmed CloudZero mode is configured the same way the deploy notebooks check it.
- Drove a conversation that delegated to each specialist; their model calls went to CloudZero as `litellm_request` spans, tagged with your `attendee_id`.
- Found your own events in the shared workshop org by time and model, and confirmed them by their Attendee Id.
- Saw that CloudWatch still holds every trace and log, including the orchestrator's, which CloudZero mode does not cover.

**Next:** [`1_agent_evaluations.ipynb`](../l7-agentic-operations/1_agent_evaluations.ipynb). Score how well the agents handle conversations.

---
## Cleanup

There is **nothing to tear down**. This notebook created no resources: it read SSM and CloudWatch Logs and drove traffic through the already-deployed agents. CloudZero is SaaS, and the workshop organization is managed by the workshop team.

**Do not delete** the `config/cloudzero/*` or `config/observability/provider` SSM parameters here - they are owned by the L0 selector and read by the L3 deploys. To stop sending model calls to CloudZero, pick CloudWatch in the Partner Selection UI, then re-run `2_specialist_agents.ipynb`.

In [ ]:
# === CLEANUP ===
# Nothing to delete here. This notebook provisioned no resources - it only read
# SSM and CloudWatch Logs and drove traffic through the already-deployed agents.
#
# Do NOT delete /anycompany/agentcore/config/cloudzero/* or
# /anycompany/agentcore/config/observability/provider here - those params are
# owned by L0 and read by the L3 deploy notebooks.
print("Nothing to clean up - CloudZero is SaaS and this notebook created no resources.")